# openEO backup — susceptibility predictor layers

This notebook demonstrates that the main **data sources** behind Practical 06 are available natively through CDSE openEO:

- Sentinel-2 for typical summer NDMI;
- ESA WorldCover 2021;
- Copernicus DEM GLO-30;
- native `slope` and `aspect` processes.

It intentionally stops at the predictor layers. The canonical weighted-overlay model in Practical 06 remains the reference susceptibility calculation.

Why stop here? Re-implementing every rescaling, class remap and weighting rule in a second backend would add maintenance cost without improving the learning objective. If GEE is unavailable, use these predictor layers together with the trainer reference susceptibility output.

### Kernel

Run this fallback in the **same Python 3 kernel used for the canonical GEE notebooks**.

The cell below installs only the lightweight `openeo` Python client when it is missing. Do not install GeoPandas, Rasterio, xarray or other compiled geospatial packages into CDSE's dedicated OpenEO kernel just for this course.

In [ ]:
import importlib.util
import subprocess
import sys

# The openEO client is intentionally the only package installed at runtime.
# The canonical Python 3 course kernel already provides the geospatial stack.
if importlib.util.find_spec("openeo") is None:
    print("Installing the openEO Python client in the current session...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openeo>=0.50,<1",
    ])

required_existing = ["geopandas", "numpy", "matplotlib", "xarray", "netCDF4"]
missing_existing = [
    name for name in required_existing
    if importlib.util.find_spec(name) is None
]

if missing_existing:
    raise RuntimeError(
        "This fallback should run in the same Python 3 kernel as the GEE "
        "practicals. Missing core package(s): "
        + ", ".join(missing_existing)
        + ". Do not repair this by installing compiled geospatial packages "
        "into the dedicated OpenEO kernel; switch to the course Python 3 kernel."
    )

print("Fallback kernel ready.")

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import openeo
import xarray as xr

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd(),
        Path.home() / "mystorage" / "fire-school",
    ]
    for candidate in candidates:
        if (candidate / "data/effis/Galicica.gpkg").exists():
            return candidate
    raise FileNotFoundError("Could not find the fire-school repository.")

REPO_ROOT = find_repo_root()
effis = gpd.read_file(REPO_ROOT / "data/effis/Galicica.gpkg").to_crs("EPSG:4326")

target = effis[effis["id"].astype(str) == "240575"].copy()
if target.empty:
    raise RuntimeError("EFFIS target polygon 240575 was not found.")

west, south, east, north = target.total_bounds
BUFFER_DEG = 0.04

BBOX = {
    "west": float(west - BUFFER_DEG),
    "south": float(south - BUFFER_DEG),
    "east": float(east + BUFFER_DEG),
    "north": float(north + BUFFER_DEG),
    "crs": "EPSG:4326",
}

print("Backup extent:", BBOX)

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("openEO authenticated.")

## 1. Typical summer NDMI, 2019–2023

Each year's July–August observations are composited first. NDMI is calculated for each annual composite, then the five annual NDMI layers are reduced with a median.

In [ ]:
YEARS = list(range(2019, 2024))
intervals = [[f"{year}-07-01", f"{year}-09-01"] for year in YEARS]
labels = [f"{year}-08-01" for year in YEARS]
MAX_CLOUD = 80

scl = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=BBOX,
    temporal_extent=[intervals[0][0], intervals[-1][1]],
    bands=["SCL"],
    max_cloud_cover=MAX_CLOUD,
)

cloud_mask = scl.process(
    "to_scl_dilation_mask",
    data=scl,
    kernel1_size=17,
    kernel2_size=77,
    mask1_values=[2, 4, 5, 6, 7],
    mask2_values=[3, 8, 9, 10, 11],
    erosion_kernel_size=3,
)

s2 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=BBOX,
    temporal_extent=[intervals[0][0], intervals[-1][1]],
    bands=["B08", "B11"],
    max_cloud_cover=MAX_CLOUD,
)

summer_bands = (
    s2
    .mask(cloud_mask)
    .aggregate_temporal(
        intervals=intervals,
        reducer="median",
        labels=labels,
    )
)

nir = summer_bands.band("B08")
swir1 = summer_bands.band("B11")
ndmi_by_year = (nir - swir1) / (nir + swir1)

typical_ndmi = (
    ndmi_by_year
    .reduce_temporal("median")
    .add_dimension(
        name="bands",
        label="typical_NDMI",
        type="bands",
    )
)

print("Typical-NDMI graph ready.")

## 2. WorldCover, slope and aspect

In [ ]:
worldcover = (
    connection
    .load_collection(
        "ESA_WORLDCOVER_10M_2021_V2",
        spatial_extent=BBOX,
    )
    .reduce_temporal("max")
)

dem = (
    connection
    .load_collection(
        "COPERNICUS_30",
        spatial_extent=BBOX,
    )
    .reduce_temporal("max")
)

slope = dem.process("slope", data=dem)
aspect = dem.process("aspect", data=dem)

# Align the static predictors to the Sentinel-2 NDMI grid.
worldcover_aligned = (
    worldcover
    .resample_cube_spatial(typical_ndmi, method="near")
    .rename_labels("bands", ["WorldCover"])
)

slope_aligned = (
    slope
    .resample_cube_spatial(typical_ndmi, method="bilinear")
    .rename_labels("bands", ["slope"])
)

aspect_aligned = (
    aspect
    .resample_cube_spatial(typical_ndmi, method="bilinear")
    .rename_labels("bands", ["aspect"])
)

bundle = (
    typical_ndmi
    .merge_cubes(worldcover_aligned)
    .merge_cubes(slope_aligned)
    .merge_cubes(aspect_aligned)
)

print("Predictor bundle ready.")

## 3. Execute the compact predictor bundle

In [ ]:
OUTPUT_DIR = Path("/tmp/geo_adapt_openeo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / "susceptibility_predictors.nc"

try:
    bundle.download(str(OUTPUT), format="NetCDF")
    print("Synchronous result:", OUTPUT)
except Exception as exc:
    print("Synchronous request did not finish:", exc)
    print("Switching to a batch job...")
    bundle.execute_batch(
        outputfile=str(OUTPUT),
        out_format="NetCDF",
        title="GEO-ADAPT openEO backup: susceptibility predictors",
    )
    print("Batch result:", OUTPUT)

## 4. Inspect the four predictors

In [ ]:
EXPECTED_BANDS = [
    ("typical_NDMI", "Typical summer NDMI"),
    ("WorldCover", "WorldCover class"),
    ("slope", "Slope"),
    ("aspect", "Aspect"),
]

ds = xr.load_dataset(OUTPUT, engine="netcdf4")
print(ds)

def extract_band(dataset, expected_name, fallback_index):
    if expected_name in dataset.data_vars:
        return dataset[expected_name].squeeze(drop=True)

    variables = list(dataset.data_vars)

    if len(variables) == 1:
        da = dataset[variables[0]]
        band_dim = next(
            (d for d in da.dims if d.lower() in {"band", "bands"}),
            None,
        )
        if band_dim is not None and da.sizes[band_dim] > fallback_index:
            return da.isel({band_dim: fallback_index}).squeeze(drop=True)

    if fallback_index < len(variables):
        return dataset[variables[fallback_index]].squeeze(drop=True)

    raise RuntimeError(
        f"Could not locate '{expected_name}' in NetCDF variables {variables}."
    )

for i, (expected, title) in enumerate(EXPECTED_BANDS):
    da = extract_band(ds, expected, i)
    arr = np.asarray(da.values, dtype="float32")
    arr[~np.isfinite(arr)] = np.nan

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(np.squeeze(arr))
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

    print(
        title,
        "| min:", round(float(np.nanmin(arr)), 3),
        "median:", round(float(np.nanmedian(arr)), 3),
        "max:", round(float(np.nanmax(arr)), 3),
    )

ds.close()

## Interpretation

Use these layers to discuss the same assumptions as in Practical 06:

- WorldCover is a broad vegetation/fuel-type proxy, not a fuel model.
- NDMI is a vegetation-moisture proxy, not dead-fuel moisture or fuel load.
- Slope and aspect influence terrain and exposure but do not determine ignition.
- A weighted susceptibility map depends on explicit modelling choices.

For the final susceptibility classes and weight-sensitivity exercise, use the canonical Practical 06 result or the trainer reference output.